# BERT sentence-pair classification

Fine-tunes `bert-base-uncased` on (review, aspect) pairs. The learning rate is chosen on the validation split, the weights of the best epoch are kept, and the official test file is scored once. Training uses bfloat16 on the GPU and takes a few minutes per domain on an A100.

## Setup

Run this on a Colab GPU runtime (*Runtime → Change runtime type → GPU*). The cell clones the repository, installs the dependencies and checks the environment. Colab keeps its own `torch`, so it is left as it is.

In [ ]:
import os

if not os.path.isdir("aspect-based-sentiment-analysis"):
    !git clone -q https://github.com/Shivam-Shukla0/aspect-based-sentiment-analysis.git
%cd aspect-based-sentiment-analysis
!pip install -q -r requirements.txt
!pip install -q -e . --no-deps
!python -m scripts.verify_environment --require-gpu

## Data

The official SemEval-2014 Task 4 files are distributed through the ILSP Metashare repository and need an account and acceptance of its licence (see `data/README.md`). Download them once and keep them in Google Drive; this cell copies them into `data/raw/` and builds the processed record files. Nothing in `data/` is committed to Git.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = Path("/content/drive/MyDrive/absa/semeval2014")  # change to where you stored the files
FILES = {
    "Laptop_Train_v2.xml": ("laptop", "train"),
    "Restaurants_Train_v2.xml": ("restaurant", "train"),
    "Laptops_Test_Gold.xml": ("laptop", "test"),
    "Restaurants_Test_Gold.xml": ("restaurant", "test"),
}
Path("data/raw").mkdir(parents=True, exist_ok=True)
for name, (domain, split) in FILES.items():
    !cp "{DRIVE_DIR / name}" data/raw/
    !python -m scripts.build_processed data/raw/{name} --domain {domain} --split {split}

## Fine-tune and evaluate

In [ ]:
for domain in ("laptop", "restaurant"):
    !python -m scripts.train_bert --domain {domain} --output-dir outputs/bert_{domain} 2>&1 | grep -E "^(run|laptop|restaurant)"

## Results

In [ ]:
import json

import pandas as pd

rows = []
for domain in ("laptop", "restaurant"):
    summary = json.load(open(f"outputs/bert_{domain}/summary.json"))
    report = json.load(open(f"outputs/bert_{domain}/test_report.json"))
    print(domain, "- validation macro F1 per setting:")
    for run in summary["runs"]:
        print("   ", {k: v for k, v in run.items() if k != "config"} | {k: run["config"][k] for k in ("hidden_size", "learning_rate", "dropout") if "config" in run})
    for cls in report["metrics"]["per_class"]:
        rows.append({"domain": domain, **{k: cls[k] for k in ("label", "precision", "recall", "f1", "support")}})
    print(f"   TEST accuracy {summary['test_accuracy']:.3f}  macro F1 {summary['test_macro_f1']:.3f}  ({summary['test_records']} records)")
pd.DataFrame(rows).round(3)

## Confusion matrices

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, domain in zip(axes, ("laptop", "restaurant")):
    report = json.load(open(f"outputs/bert_{domain}/test_report.json"))
    matrix = np.array(report["confusion_matrix"])
    ax.imshow(matrix, cmap="Blues")
    ax.set_xticks(range(len(report["labels"])), report["labels"])
    ax.set_yticks(range(len(report["labels"])), report["labels"])
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(domain)
    for (i, j), value in np.ndenumerate(matrix):
        ax.text(j, i, int(value), ha="center", va="center")
plt.tight_layout()
plt.show()

## Where the errors are

Counts of (true → predicted) pairs, which show which polarities the model confuses.

In [ ]:
for domain in ("laptop", "restaurant"):
    errors = json.load(open(f"outputs/bert_{domain}/test_report.json"))["errors"]
    print(domain, errors["total_errors"], "errors")
    for pair, count in sorted(errors["counts_by_pair"].items(), key=lambda kv: -kv[1]):
        print(f"   {pair:24s} {count}")